In [32]:
import json
import pandas as pd

In [33]:
# Load cleaned JSON
with open("../data/springfield_locations_cleaned.json", "r", encoding="utf-8") as f:
    locations = json.load(f)

In [34]:
df = pd.DataFrame(locations)

In [35]:
df.head()

,location_name,location_type,description,canonical_characters,key_activities,notable_events,relevance_to_the_simpson_family,parody_of,history,relevance_to_other_characters,additional_notes,url
0,$50 Marmalade,Retail Shop,$50 Marmalade is a shop located in the promena...,[],"[Shopping for gourmet food items, Exploring un...",[],[The shop's location in Springfield Heights in...,NaN,NaN,NaN,NaN,NaN
1,All Night Gym,Gym,A 24-hour fitness facility located in Springfi...,"[Homer Simpson, Rainier Wolfcastle]","[Physical fitness workouts, Personal training ...","[Homer's attempts to get fit, Visiting celebri...","[Homer's misguided efforts to lose weight, Rep...",NaN,NaN,NaN,NaN,NaN
2,Alkali Flats,Natural landmark,"The Alkali Flats, also known as the Springfiel...","[Krusty the Clown, Chief Wiggum, Homer Simpson...","[Broadcasting television shows, Adventure expl...",[Krusty broadcasts from the Alkali Flats durin...,[Homer and Chief Wiggum's adventures showcase ...,NaN,NaN,NaN,NaN,NaN
3,All Creatures Great and Cheap,Pet Store,A pet store located inside the Springfield Mal...,"[Homer Simpson, Lisa Simpson, Bart Simpson]","[Pet purchases, Pet interactions, Animal quarr...","[Homer argues with a pet bird, Lisa buys a ham...",[Homer frequently visits to interact with pets...,NaN,NaN,NaN,NaN,NaN
4,All Trees $75,Retail,This establishment in Springfield specializes ...,"[Homer Simpson, Marge Simpson, Bart Simpson, L...","[Purchasing Christmas trees, Decorating for th...",[Homer's search for a Christmas tree after los...,[The shop exemplifies the financial struggles ...,NaN,NaN,NaN,NaN,NaN


In [36]:
from sentence_transformers import SentenceTransformer

In [37]:
# A a small, fast local model
# model = SentenceTransformer('all-MiniLM-L6-v2')

# Load a high-quality local model
model = SentenceTransformer("all-mpnet-base-v2") 


In [38]:
# Generate embeddings for each location description
descriptions = df['description'].fillna("").astype(str).tolist() # ensure no NaNs
embeddings = model.encode(descriptions, convert_to_tensor=True)

In [39]:
import faiss
import numpy as np

In [40]:
# Convert embeddings to numpy float32
embeddings_np = embeddings.cpu().detach().numpy().astype('float32')

In [41]:
# Create FAISS index
dim = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings_np)

In [42]:
def ask_question(question, top_k=3):
    # Embed the question
    q_emb = model.encode([question]).astype('float32')
    
    # Search the index
    D, I = index.search(q_emb, top_k)
    
    # Return the top matches
    results = []
    for i in I[0]:
        results.append({
            "location_name": df.iloc[i]['location_name'],
            "location_type": df.iloc[i]['location_type'],
            "description": df.iloc[i]['description']
        })
    return results

In [50]:
# Example
question = "Tell me about Bowlerama."
answers = ask_question(question)
for a in answers:
    print(f"{a['location_name']} ({a['location_type']}): {a['description']}\n")

Barney's Bowlarama (Bowling Alley): A popular bowling alley in Springfield, Barney's Bowlarama is owned by Al Gumble, the uncle of notable resident Barney Gumble. It has been an important recreational venue in the community, especially for the local bowling leagues and casual outings among Springfield's residents. The Bowlarama has endured significant challenges over the years, including destruction by fire during a major downtown incident that also claimed nearby landmarks such as the Springfield Symphony Hall and the Springfield Museum of Natural History. The bowling alley was rebuilt shortly after the fire, demonstrating the resilience and dedication of the community towards maintaining local entertainment venues. Additionally, the Bowlarama has been relocated to the top of a mountain following a devastating hurricane that left extensive damage across Springfield.

Nick's Bowling Shop (Bowling shop): Nick's Bowling Shop is one of the prominent bowling establishments in Springfield, 